In [75]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
import os
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import database

In [76]:
API_KEY = os.environ['KEY']

In [77]:
SERIES_IDS = ['UNRATE', 'INDPRO', 'JTSJOL']
feature_set = ['log_10_JTSJOL_t-1', 'UNRATE_t', 'INDPRO_t-1']

# Get latest observation dates for each series ID from on FRED

In [78]:
with requests.Session() as session:

    url = "https://api.stlouisfed.org/fred/series"
    latest_obs_date_fred = {}

    for ID in SERIES_IDS:
        params = {"api_key": API_KEY,
                  "file_type": "json",
                  "series_id": ID}

        r = session.get(url=url, params=params, timeout=30)
        r.raise_for_status()

        data = r.json()

        latest_obs = data['seriess'][0]['observation_end']

        latest_obs_date_fred.update({ID: latest_obs})

    latest_obs_date_fred = {
    series_id: datetime.fromisoformat(date)
    for series_id, date in latest_obs_date_fred.items()
    }


In [79]:
latest_obs_date_fred

{'UNRATE': datetime.datetime(2026, 8, 1, 0, 0),
 'INDPRO': datetime.datetime(2026, 7, 1, 0, 0),
 'JTSJOL': datetime.datetime(2026, 7, 1, 0, 0)}

In [80]:
type(latest_obs_date_fred['UNRATE'])

datetime.datetime

# Get dates of latest observation entry for each series ID from db
Table: `FREDSeries`; Column: `ObservationEnd`

In [81]:
latest_obs_date_db = {}
db = database.get_latest_obs_end(series_ids=SERIES_IDS)
latest_obs_date_db = {
    series_id: datetime.fromisoformat(date)
    for series_id, date in db
}

In [82]:
latest_obs_date_db

{'INDPRO': datetime.datetime(2026, 7, 1, 0, 0),
 'JTSJOL': datetime.datetime(2026, 7, 1, 0, 0),
 'UNRATE': datetime.datetime(2026, 8, 1, 0, 0)}

In [83]:
type(latest_obs_date_db['UNRATE'])

datetime.datetime

# Compare dates

In [ ]:
# latest_obs_date_fred = {
#     'UNRATE': datetime(2026, 9, 1, 0, 0),
#     'INDPRO': datetime(2026, 8, 1, 0, 0),
#     'JTSJOL': datetime(2026, 8, 1, 0, 0),
# }

In [93]:
latest_obs_date_fred

{'UNRATE': datetime.datetime(2026, 9, 1, 0, 0),
 'INDPRO': datetime.datetime(2026, 8, 1, 0, 0),
 'JTSJOL': datetime.datetime(2026, 8, 1, 0, 0)}

In [92]:
all_db_obs_dates_outdated = all(
    ID not in latest_obs_date_db
     or latest_obs_date_fred[ID] > latest_obs_date_db[ID]
    for ID in latest_obs_date_db
)
all_db_obs_dates_outdated

True

# Final code

In [68]:
# --- returns true if new observations for all series IDs are available on FRED ---

def new_obs_available(series_ids):

    # get latest observation dates of series IDs from FRED
    with requests.Session() as session:

        url = "https://api.stlouisfed.org/fred/series"
        latest_obs_date_fred = {}

        for ID in SERIES_IDS:
            params = {"api_key": API_KEY,
                    "file_type": "json",
                    "series_id": ID}

            r = session.get(url=url, params=params, timeout=30)
            r.raise_for_status()

            data = r.json()

            latest_obs = data['seriess'][0]['observation_end']

            latest_obs_date_fred.update({ID: latest_obs})

        latest_obs_date_fred = {
        series_id: datetime.fromisoformat(date)
        for series_id, date in latest_obs_date_fred.items()
        }

    # get latest observation dates of series IDs from db
    latest_obs_date_db = {}
    db = database.get_latest_obs_end(series_ids=SERIES_IDS)
    latest_obs_date_db = {
        series_id: datetime.fromisoformat(date)
        for series_id, date in db
    }

    # compare if for all series IDs are new observations available on FRED
    all_db_obs_dates_outdated = all(
    ID not in latest_obs_date_db
     or latest_obs_date_fred[ID] > latest_obs_date_db[ID]
    for ID in latest_obs_date_db
    )

    return all_db_obs_dates_outdated


In [71]:
print(f"latest_obs_date_db:\n{latest_obs_date_db}\n\nlatest_obs_date_fred:\n{latest_obs_date_fred}")

latest_obs_date_db:
{'INDPRO': datetime.datetime(2026, 7, 1, 0, 0), 'JTSJOL': datetime.datetime(2026, 7, 1, 0, 0), 'UNRATE': datetime.datetime(2026, 8, 1, 0, 0)}

latest_obs_date_fred:
{'UNRATE': datetime.datetime(2026, 8, 1, 0, 0), 'INDPRO': datetime.datetime(2026, 7, 1, 0, 0), 'JTSJOL': datetime.datetime(2026, 7, 1, 0, 0)}


In [70]:
new_obs_available(SERIES_IDS)

False